install and import libraries

In [4]:
#pip install pandas

In [5]:
import pandas as pd
import glob
import sqlite3

biometric data

In [6]:
bio_files = glob.glob("../data/raw/biometric/*.csv")
biometric_df = pd.concat(
    (pd.read_csv(f) for f in bio_files),
    ignore_index=True
)
print("Biometric rows:", biometric_df.shape)

Biometric rows: (1861108, 6)


In [7]:
biometric_df.head()

,date,state,district,pincode,bio_age_5_17,bio_age_17_
0,01-03-2025,Haryana,Mahendragarh,123029,280,577
1,01-03-2025,Bihar,Madhepura,852121,144,369
2,01-03-2025,Jammu and Kashmir,Punch,185101,643,1091
3,01-03-2025,Bihar,Bhojpur,802158,256,980
4,01-03-2025,Tamil Nadu,Madurai,625514,271,815


In [9]:
biometric_df.isnull().sum()

date            0
state           0
district        0
pincode         0
bio_age_5_17    0
bio_age_17_     0
dtype: int64

Demographic data

In [10]:
demo_files = glob.glob("../data/raw/demographic/*.csv")
demographic_df = pd.concat(
    (pd.read_csv(f) for f in demo_files),
    ignore_index=True
)
print("Demographic rows:", demographic_df.shape)

Demographic rows: (2071700, 6)


In [11]:
demographic_df.head()

,date,state,district,pincode,demo_age_5_17,demo_age_17_
0,01-03-2025,Uttar Pradesh,Gorakhpur,273213,49,529
1,01-03-2025,Andhra Pradesh,Chittoor,517132,22,375
2,01-03-2025,Gujarat,Rajkot,360006,65,765
3,01-03-2025,Andhra Pradesh,Srikakulam,532484,24,314
4,01-03-2025,Rajasthan,Udaipur,313801,45,785


In [13]:
demographic_df.isnull().sum()

date             0
state            0
district         0
pincode          0
demo_age_5_17    0
demo_age_17_     0
dtype: int64

Enrolment data

In [14]:
enr_files = glob.glob("../data/raw/enrolment/*.csv")
enrolment_df = pd.concat(
    (pd.read_csv(f) for f in enr_files),
    ignore_index=True
)
print("Enrolment rows:", enrolment_df.shape)

Enrolment rows: (1006029, 7)


In [15]:
enrolment_df.head()

,date,state,district,pincode,age_0_5,age_5_17,age_18_greater
0,02-03-2025,Meghalaya,East Khasi Hills,793121,11,61,37
1,09-03-2025,Karnataka,Bengaluru Urban,560043,14,33,39
2,09-03-2025,Uttar Pradesh,Kanpur Nagar,208001,29,82,12
3,09-03-2025,Uttar Pradesh,Aligarh,202133,62,29,15
4,09-03-2025,Karnataka,Bengaluru Urban,560016,14,16,21


In [17]:
enrolment_df.isnull().sum()

date              0
state             0
district          0
pincode           0
age_0_5           0
age_5_17          0
age_18_greater    0
dtype: int64

Standardizing the column names

Check for duplicates

In [18]:
print("Biometric duplicates:", biometric_df.duplicated().sum())
print("Demographic duplicates:", demographic_df.duplicated().sum())
print("Enrolment duplicates:", enrolment_df.duplicated().sum())

Biometric duplicates: 94896
Demographic duplicates: 473601
Enrolment duplicates: 22957


In [19]:
def clean_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df

biometric_df = clean_columns(biometric_df)
demographic_df = clean_columns(demographic_df)
enrolment_df = clean_columns(enrolment_df)

sql

In [20]:
conn = sqlite3.connect("../data/processed/uidai.db")

biometric_df.to_sql("biometric_updates", conn, if_exists="replace", index=False)
demographic_df.to_sql("demographic_updates", conn, if_exists="replace", index=False)
enrolment_df.to_sql("enrolments", conn, if_exists="replace", index=False)

1006029

In [21]:
pd.read_sql(
    "SELECT state, COUNT(*) AS total FROM biometric_updates GROUP BY state",
    conn
)


,state,total
0,Andaman & Nicobar Islands,549
1,Andaman and Nicobar Islands,1298
2,Andhra Pradesh,172034
3,Arunachal Pradesh,4244
4,Assam,47643
5,Bihar,83398
6,Chandigarh,1656
7,Chhatisgarh,5
8,Chhattisgarh,31992
9,Dadra & Nagar Haveli,100


In [22]:
pd.read_sql("""
SELECT state, COUNT(*) AS records
FROM enrolments
GROUP BY state
ORDER BY records DESC
LIMIT 5
""", conn)

,state,records
0,Uttar Pradesh,110369
1,Tamil Nadu,92552
2,Maharashtra,77191
3,West Bengal,76519
4,Karnataka,70198


Saving of cleaned data

In [24]:
biometric_df.to_csv("../data/processed/biometric_clean.csv", index=False)
demographic_df.to_csv("../data/processed/demographic_clean.csv", index=False)
enrolment_df.to_csv("../data/processed/enrolment_clean.csv", index=False)